In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")


In [4]:
from datasets import load_dataset

data = load_dataset('aeirya/lct-mt',
                    'kor', 
                    token = HF_TOKEN,
                    #streaming=True,
                    split = "train",
                    data_files = ["kor/train-00000-of-00010.parquet"]
                    )


Generating train split:   0%|          | 0/28849814 [00:00<?, ? examples/s]

NonMatchingSplitsSizesError: [{'expected': SplitInfo(name='train', num_bytes=4899528894, num_examples=28849814, shard_lengths=None, dataset_name=None), 'recorded': SplitInfo(name='train', num_bytes=489884103, num_examples=2884982, shard_lengths=None, dataset_name='lct-mt')}]

In [ ]:
print(data)


In [ ]:
print(data["train"])


In [ ]:

print(data["train"][0])


In [ ]:
#to see the language options:
from datasets import get_dataset_config_names

get_dataset_config_names("aeirya/lct-mt",
                         token = HF_TOKEN)


In [ ]:
# Prepare thedata
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq 

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M",
                                             # dtype="auto",
                                              attn_implementation="sdpa")

tokenizer.src_lang = "eng_Latn"
tokenizer.tgt_lang = ""

def tokenize_function(data):
    model_inputs = tokenizer(
        data["eng"],
        truncation=True
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            data["text"],
            truncation=True
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = data.map(tokenize_function,
                              batched=True)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model = model)

print(tokenized_datasets)


In [ ]:
#tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
#tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names()


In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    batch_size=8,
    collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=8,
    collate_fn=data_collator
)



In [ ]:
# Check format
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}


In [ ]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)


In [ ]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)


In [ ]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device


In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)



In [ ]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()
